# Musketeer Chess NNUE — Train all 4 models on a Colab GPU

This notebook trains all four Musketeer NNUE models on Google Colab's free GPU.

**Before you start:** in Colab, set **Runtime → Change runtime type → GPU**.

**What you need:** the file `Musketeer_Colab_bundle.zip` (contains the code and the
6 self-play PGNs). You'll upload it in Step 2.

The training scripts auto-detect the GPU — no code changes needed.

## Step 1 — Confirm the GPU is available

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU! Go to Runtime -> Change runtime type -> GPU, then re-run.')

## Step 2 — Upload and unpack the project bundle
Run this cell, then click **Choose Files** and pick `Musketeer_Colab_bundle.zip`.
(Alternatively, put the zip in Google Drive and use the commented Drive lines.)

In [ ]:
import os, zipfile, glob

# --- Option A: direct upload (default) ---
from google.colab import files
up = files.upload()                     # pick Musketeer_Colab_bundle.zip
zip_name = next(iter(up))

# --- Option B: from Google Drive (uncomment to use instead) ---
# from google.colab import drive
# drive.mount('/content/drive')
# zip_name = '/content/drive/MyDrive/Musketeer_Colab_bundle.zip'

os.makedirs('/content/musketeer', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/musketeer')
os.chdir('/content/musketeer')
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Project root:', os.getcwd())
print('PGNs:', [os.path.basename(p) for p in glob.glob('data/raw/*.pgn')])
print('Code :', sorted(os.path.basename(p) for p in glob.glob('src/*.py') + glob.glob('train/*.py')))

## Step 3 — Rebuild the training database from the PGNs
Pure-Python parser (no engine needed): replays every game to `(fen, eval, result)`.
Takes a couple of minutes for all ~5,900 games / 835k positions.

In [ ]:
import glob, subprocess, os
for pgn in sorted(glob.glob('data/raw/*.pgn')):
    out = 'data/processed/' + os.path.basename(pgn).replace('.pgn','').replace(' ','_')
    print('parsing', os.path.basename(pgn), '...')
    subprocess.run(['python','src/pgn_to_fen.py', pgn,
                    '--jsonl', out+'.jsonl', '--plain', out+'.plain'], check=True)
n = sum(1 for f in glob.glob('data/processed/*training*.jsonl') for _ in open(f))
print('total training positions:', n)

## Step 4 — Train all four models on the GPU
The first model encodes the dataset and caches it (`.npz`); the others reuse the
cache. Adjust `EPOCHS` for longer/shorter runs. `DATA` globs all training files.

In [ ]:
DATA = 'data/processed/uni_hawk_training_nnue-*.jsonl'
EPOCHS = 30
BATCH = 16384    # large batch is fine on GPU

!python train/model1.py --data "$DATA" --epochs $EPOCHS --batch $BATCH --out models/model1.pt
!python train/model2.py --data "$DATA" --epochs $EPOCHS --batch $BATCH --out models/model2.pt
!python train/model3.py --data "$DATA" --epochs $EPOCHS --batch $BATCH --out models/model3.pt
!python train/model4.py --data "$DATA" --epochs $EPOCHS --batch $BATCH --out models/model4.pt

## Step 5 — Compare the four trained models on one split

In [ ]:
!python train/compare.py --data "$DATA" --limit 200000 --epochs $EPOCHS

## Step 6 — Download the trained models

In [ ]:
import zipfile, glob
with zipfile.ZipFile('trained_models.zip','w') as z:
    for p in glob.glob('models/*.pt'):
        z.write(p)
from google.colab import files
files.download('trained_models.zip')
# Or save to Drive:  !cp trained_models.zip /content/drive/MyDrive/